## Notebook 1: Data Exploration

**Goal:** Explore the Fitbit dataset to understand its structure, quality, 
and key variables before building our analysis pipeline.

**Datasets used:**
- `dailyActivity_merged.csv` — steps, calories, active minutes per user per day
- `heartrate_seconds_merged.csv` — raw heart rate readings
- `minuteSleep_merged.csv` — minute-level sleep data
- `weightLogInfo_merged.csv` — weight and BMI logs

## 1. Environment Setup
Setting up Java environment and initializing the Spark session.

In [ ]:
import os
os.environ["JAVA_HOME"] = "/opt/homebrew/opt/openjdk@17"
os.environ["PATH"] = "/opt/homebrew/opt/openjdk@17/bin:" + os.environ["PATH"]

from pyspark.sql import SparkSession
import pyspark.sql.functions as F
import pandas as pd

# Start Spark session
spark = SparkSession.builder \
    .appName("WearableHealthAnalyzer") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR") 

print("Spark version:", spark.version)

Spark version: 4.1.1


## 2. Load Data

Loading all four datasets from both time periods into Spark DataFrames.

In [ ]:
BASE_PATH_1 = "../data/archive (3)/mturkfitbit_export_3.12.16-4.11.16/Fitabase Data 3.12.16-4.11.16/"
BASE_PATH_2 = "../data/archive (3)/mturkfitbit_export_4.12.16-5.12.16/Fitabase Data 4.12.16-5.12.16/"

def load_csv(filename):
    """Load a CSV from both time periods and combine into one DataFrame."""
    df1 = spark.read.csv(BASE_PATH_1 + filename, header=True, inferSchema=True)
    df2 = spark.read.csv(BASE_PATH_2 + filename, header=True, inferSchema=True)
    return df1.union(df2)

# Load all four datasets
daily_activity = load_csv("dailyActivity_merged.csv")
heart_rate     = load_csv("heartrate_seconds_merged.csv")
sleep          = load_csv("minuteSleep_merged.csv")
weight         = load_csv("weightLogInfo_merged.csv")

## 3. Data Exploration

### 3.1 Daily Activity

In [7]:
print("Shape:", (daily_activity.count(), len(daily_activity.columns)))
print("Unique users:", daily_activity.select("Id").distinct().count())
print("\nSchema:")
daily_activity.printSchema()

Shape: (1397, 15)
Unique users: 35

Schema:
root
 |-- Id: long (nullable = true)
 |-- ActivityDate: string (nullable = true)
 |-- TotalSteps: integer (nullable = true)
 |-- TotalDistance: double (nullable = true)
 |-- TrackerDistance: double (nullable = true)
 |-- LoggedActivitiesDistance: double (nullable = true)
 |-- VeryActiveDistance: double (nullable = true)
 |-- ModeratelyActiveDistance: double (nullable = true)
 |-- LightActiveDistance: double (nullable = true)
 |-- SedentaryActiveDistance: double (nullable = true)
 |-- VeryActiveMinutes: integer (nullable = true)
 |-- FairlyActiveMinutes: integer (nullable = true)
 |-- LightlyActiveMinutes: integer (nullable = true)
 |-- SedentaryMinutes: integer (nullable = true)
 |-- Calories: integer (nullable = true)



In [8]:
print("Sample rows:")
daily_activity.show(5)

# Summary statistics for key columsn
print("Summary statistics:")
daily_activity.select(
    "TotalSteps",
    "Calories",
    "VeryActiveMinutes",
    "SedentaryMinutes"
).describe().show()

Sample rows:
+----------+------------+----------+----------------+----------------+------------------------+------------------+------------------------+-------------------+-----------------------+-----------------+-------------------+--------------------+----------------+--------+
|        Id|ActivityDate|TotalSteps|   TotalDistance| TrackerDistance|LoggedActivitiesDistance|VeryActiveDistance|ModeratelyActiveDistance|LightActiveDistance|SedentaryActiveDistance|VeryActiveMinutes|FairlyActiveMinutes|LightlyActiveMinutes|SedentaryMinutes|Calories|
+----------+------------+----------+----------------+----------------+------------------------+------------------+------------------------+-------------------+-----------------------+-----------------+-------------------+--------------------+----------------+--------+
|1503960366|   3/25/2016|     11004| 7.1100001335144| 7.1100001335144|                     0.0|   2.5699999332428|        0.46000000834465|   4.07000017166138|                    0

### 3.2 Heart Rate, Sleep & Weight

In [9]:
print("Heart Rate rows:", heart_rate.count(), "| Unique users:", heart_rate.select("Id").distinct().count())
heart_rate.show(3)

print("Sleep rows:", sleep.count(), "| Unique users:", sleep.select("Id").distinct().count())
sleep.show(3)

print("Weight rows:", weight.count(), "| Unique users:", weight.select("Id").distinct().count())
weight.show(3)

Heart Rate rows: 3638339 | Unique users: 15
+----------+-------------------+-----+
|        Id|               Time|Value|
+----------+-------------------+-----+
|2022484408|4/1/2016 7:54:00 AM|   93|
|2022484408|4/1/2016 7:54:05 AM|   91|
|2022484408|4/1/2016 7:54:10 AM|   96|
+----------+-------------------+-----+
only showing top 3 rows
Sleep rows: 387080 | Unique users: 25
+----------+--------------------+-----+-----------+
|        Id|                date|value|      logId|
+----------+--------------------+-----+-----------+
|1503960366|3/13/2016 2:39:30 AM|    1|11114919637|
|1503960366|3/13/2016 2:40:30 AM|    1|11114919637|
|1503960366|3/13/2016 2:41:30 AM|    1|11114919637|
+----------+--------------------+-----+-----------+
only showing top 3 rows
Weight rows: 100 | Unique users: 13
+----------+--------------------+----------------+----------------+----+----------------+--------------+-------------+
|        Id|                Date|        WeightKg|    WeightPounds| Fat|      

### 3.3 Dataset Summary

- not all users tracked all metrics, analysis will work on users with complete data
- daily activity is our most complete dataset(35 users)
- weight data is too sparse for any strong conclusions